**NN TFLITE TEMPLATE WITH WINDOWED DATA**

*This notebook will pull in the class data that has been collated for sitting, standing, walking, and falling. It provides a basis for creating an ML model that will take in gyroscope/accelerometer data from the Raspberry Pi and determine what behavior is being performed.*

**Part I. Set-Up**

First, we need to make sure we are working with a version of Tensorflow that complements our capabilities on our embedded system.

The Pi runs Python 3.7. We will upgrade to use a tflite_runtime interpreter == 2.11.0 BUT we will still have to downgrade tensorflow (for model development) to 2.12.0 so that the oeprations are compatible for inference on our embedded system.

In [ ]:
!pip uninstall tensorflow
!pip install tensorflow==2.12.0


In [ ]:
#Make sure it is 2.12.0 before doing any work
import tensorflow as tf
print(tf.__version__)

Next we need to mount our Drive and map to where our data lives. Please make sure that the CSV files given in the github repository are saved in a folder of your choosing on your Google Drive. Then map to that directory. Feel free to use

```
!ls
```

 to check that your files are there!

In [ ]:
# prompt: navigate to a folder in my google drive My Drive > data > fall_data

import os
from google.colab import drive

drive.mount('/content/drive')
%cd /content/drive/My Drive/data/fall_data
!pwd


In [ ]:
!ls

**Part II. Data Exploration**

Now let's pull in the data using


```
pd.read_csv()
```
Label the columns and explore this data set.


*   How many data points are there in each?
*   Maybe find a way to plot a couple of the parameters over time to see what the "behavior" looks like?




In [ ]:
import pandas as pd

# Load the four CSV files into dataframes without the index
df_sitting = pd.read_csv('sitting_data.csv', index_col=False)
df_standing = pd.read_csv('standing_data.csv', index_col=False)
df_walking = pd.read_csv('walking_data.csv', index_col=False)
df_falling = pd.read_csv('falling_data.csv', index_col=False)

# Specify the columns that contain gyro and accel data
gyro_accel_columns = ['Gyro_X', 'Gyro_Y', 'Gyro_Z', 'Accel_X', 'Accel_Y', 'Accel_Z']  # Replace with actual column names



In [ ]:
# data exploration here.


**Part III. Tensorflow**

Now let's use NN. We will start with a simple NN that takes in raw data.

We also have to think about what libraries will be using and what versions are available for our Raspberry Pi.

In [ ]:
# Import libraries (add libraries you need here)
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout,Conv1D, MaxPooling1D, Flatten, Dense
from tensorflow.keras.utils import to_categorical
from sklearn import svm
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier

**Part IV. A More Sophisticated NN**

As we noticed, a simple NN with raw values did not perform very well. Let's try using window of time to break up our sensor data and use THOSE as inputs to our model.

In [ ]:
# Function to create a sliding window dataframe for specific columns
def create_sliding_window_df(df, columns, window_size):
    sliding_window_data = []
    for i in range(len(df) - window_size + 1):
        window = df[columns].iloc[i:i + window_size].values.flatten()
        sliding_window_data.append(window)
    return pd.DataFrame(sliding_window_data)

In [ ]:
# Define your window_size
window_size = 50

# Create new dataframes with sliding windows for gyro and accel data
df_sitting_windowed = create_sliding_window_df(df_sitting, gyro_accel_columns,window_size)
df_standing_windowed = create_sliding_window_df(df_standing, gyro_accel_columns,window_size)
df_walking_windowed = create_sliding_window_df(df_walking, gyro_accel_columns,window_size)
df_falling_windowed = create_sliding_window_df(df_falling, gyro_accel_columns,window_size)

# Add labels to the windowed dataframes
df_sitting_windowed['Label'] = 0
df_standing_windowed['Label'] = 1
df_walking_windowed['Label'] = 2
df_falling_windowed['Label'] = 3

In [ ]:
# DATAFRAME
# Concatenate all windowed dataframes into one large dataframe X
X_df = pd.concat([df_sitting_windowed, df_standing_windowed, df_walking_windowed, df_falling_windowed], ignore_index=True)

# Randomize the order of the X data
X_df = shuffle(X_df, random_state=42)
# Separate the labels into a dataframe y
y_df = X_df.pop('Label')

label_encoder = LabelEncoder()
y_encoded_df = label_encoder.fit_transform(y_df)
y_categorical_df = to_categorical(y_encoded_df, num_classes=4)  # Ensure 4 classes for one-hot encoding


In [ ]:
#NUMPY
# Concatenate all windowed dataframes into one large dataframe X
X = pd.concat([df_sitting_windowed, df_standing_windowed, df_walking_windowed, df_falling_windowed], ignore_index=True)

# Randomize the order of the X data
X = shuffle(X, random_state=42)
X_array = X.to_numpy() # convert to array

y_array = X_array[:,-1] #move label column into y_array

X_array = X_array[:, :-1] # Remove the last column

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_array)
y_categorical = to_categorical(y_encoded, num_classes=4)  # Ensure 4 classes for one-hot encoding


In [ ]:
#NUMPY
# Reshape X to be 3D [samples, window size, features]
#USE THIS FOR X_ARRAY

n_features = 6
X_reshape = X_array.reshape((X.shape[0], window_size, n_features))

# Split the data into training, testing, and validation sets
X_train, X_temp, y_train, y_temp = train_test_split(X_reshape, y_categorical, test_size=0.4, random_state=42)
X_test, X_val, y_test, y_val = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print("Data has been successfully split into training, testing, and validation sets.")
print(f"Training set size: {len(X_train)}")
print(f"Testing set size: {len(X_test)}")
print(f"Validation set size: {len(X_val)}")

# Build the CNN model
model = Sequential()
model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(window_size, n_features)))
model.add(MaxPooling1D(pool_size=2))
model.add(Flatten())
model.add(Dense(100, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(4, activation='softmax'))  # Assuming 4 classes: sitting, standing, walking, falling

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train, epochs=15, batch_size=32, validation_data=(X_val, y_val))

# Evaluate the model
loss, accuracy = model.evaluate(X_test, y_test)
print(f'Test Accuracy: {accuracy:.2f}')

In [ ]:
X_reshape.shape

In [ ]:
# DATAFRAME: DO NOT EXECUTE THIS YET
# Reshape X to be 3D [samples, window size, features]
n_features = 6 #X_array.shape[1] #features
X_reshape_df = X_df.to_numpy().reshape(X_df.shape[0], window_size, n_features) #Reshape to 3D array


# Split the data into training, testing, and validation sets
X_train_df, X_temp_df, y_train_df, y_temp_df = train_test_split(X_reshape_df, y_categorical_df, test_size=0.4, random_state=42)
X_test_df, X_val_df, y_test_df, y_val_df = train_test_split(X_temp_df, y_temp_df, test_size=0.5, random_state=42)

print("Data has been successfully split into training, testing, and validation sets.")
print(f"Training set size: {len(X_train_df)}")
print(f"Testing set size: {len(X_test_df)}")
print(f"Validation set size: {len(X_val_df)}")

input_shape = (window_size,n_features)  # Example: 10 features

# Build the CNN model
model = Sequential()
model.add(Input(shape=input_shape))
model.add(Conv1D(filters=64, kernel_size=3, activation='relu'))
model.add(MaxPooling1D(pool_size=2))
model.add(Flatten())
model.add(Dense(100, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(4, activation='softmax'))  # Assuming 4 classes: sitting, standing, walking, falling

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
history_df = model.fit(X_train_df, y_train_df, epochs=15, batch_size=32, validation_data=(X_val_df, y_val_df))

# Evaluate the model
loss_df, accuracy_df = model.evaluate(X_test_df, y_test_df)
print(f'Test Accuracy: {accuracy_df:.2f}')

In [ ]:
X_train.shape

In [ ]:
X_train_df.shape

**Part V. Optimize and Save Your Model**

At this point, we are happy with our accuracy during initial testing. We need to package our model to run on the embedded system. Similar to before, we will be saving this as a TF Lite model. NOW, we have a couple of lines that incorporate TF Lite optimization. Below is the code as well as a description of what is happening.

In [ ]:
#Set-up
model = model
model_file_name = 'nn_model_window_50np_Opt_test.tflite' #change the name for your model

# Convert the model to TensorFlow Lite format
converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,  # Built-in operations - ReLU, Conv2D, MaxPool, Dense, etc...
    tf.lite.OpsSet.SELECT_TF_OPS     # Select TensorFlow operations - maybe for LSTM/RNN layers
]
converter.optimizations = [tf.lite.Optimize.DEFAULT] # See below
converter.inference_input_type = tf.float32
converter.inference_output_type = tf.float32
tflite_model = converter.convert()

# Save the TensorFlow Lite model to a file
with open(model_file_name, 'wb') as f:
    f.write(tflite_model)

print("Model converted to TensorFlow Lite and saved as ", model_file_name)



`converter = tf.lite.TFLiteConverter.from_keras_model(model)`:

*   This initializes a TensorFlow Lite converter using the Keras model. The from_keras_model(model) method converts the Keras model into a format suitable for TensorFlow Lite.

`converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]`:

*   This specifies the operations that the TensorFlow Lite model should support.
*  `tf.lite.OpsSet.TFLITE_BUILTINS`: Built-in TensorFlow Lite operations.
*   `tf.lite.OpsSet.SELECT_TF_OPS`: Select TensorFlow operations that are not part of the built-in set but are necessary for the model.

`converter.optimizations = [tf.lite.Optimize.DEFAULT]`:
*   This sets the optimization strategy for the conversion process
*   `Optimize.DEFAULT` applies default optimizations to reduce the model size and improve performance
*   List item

`converter.inference_input_type = tf.float32`:
*   This specifies the data type for the input tensor during inference. In this case, it is set to `tf.float32` (32-bit floating point).
*   `converter.inference_output_type = tf.float32`: This specifies the data type for the output tensor during inference. It is also set to `tf.float32`.

`tflite_model = converter.convert()`:
*   This line performs the conversion of the Keras model to the TensorFlow Lite format and stores the converted model in the variable tflite_model.

https://www.tensorflow.org/api_docs/python/tf/lite/Optimize

## What does this all mean?

### With the default values:
*  Dynamic Range Quantization: This optimization reduces memory usage and speeds up computation by statically quantizing the weights from floating point to integer at conversion time. It provides 8-bit precision for the weights
*  Dynamic Quantization of Activations: During inference, the activations are dynamically quantized based on their range to 8-bits, and computations are performed with 8-bit weights and activations. This results in reduced latency, although the outputs are still stored in floating point
*   Our inputs/outputs remain float32 (easier integration with existing systems but still get the benefit of a quantized model)






**Part VI. Evaluate TF Lite Model**

Let's check the performance of our OPTIMIZED model.

In [ ]:
#Quick test of the optimized TensorFlow Lite model.
# This script loads in the saved tflite model, runs inferences using the TFLite
# interpreter and the X_test numpy array data, appends the result in y_pred.
# Then, provides the accuracy compared to y_test.


# Load the TensorFlow Lite model
interpreter = tf.lite.Interpreter(model_path="nn_model_window_50np_Opt.tflite") #change the name for your model
interpreter.allocate_tensors()

# Get input and output tensors
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

input_details = interpreter.get_input_details()
input_shape = input_details[0]['shape']

y_pred = []

for i in range(X_test.shape[0]):  # Iterate through each sample
    input_data = X_test[i:i+1]  # Get a single sample
    interpreter.set_tensor(input_details[0]['index'], input_data.astype(np.float32))
    interpreter.invoke()
    output_data = interpreter.get_tensor(output_details[0]['index'])

    predicted_class = np.argmax(output_data[0])  # Max probability
    y_pred.append(predicted_class)


# Convert y_test to a 1D array of class labels because it is one-hot encoded.
y_true = np.argmax(y_test, axis=1)

# Calculate accuracy
accuracy = accuracy_score(y_true, y_pred)
print(f"Accuracy: {accuracy}")



**Part VI. Deploy Your Model**

Below are scripts that get data from the SenseHat and package it up an appropriate window size (for this example, 50). There are two scripts however.



1.   The first script gets all 50 samples THEN performs the inference.
2.   The second script implements a sliding window which updates the data window every sample acquisition. It adds the newest entry and removes the oldest entry. Inference is happening at approximately the sampling rate.

Note the change in how we prepare the data to be fed into our models. Descriptions of these commands are below.



In [ ]:
# predict_nn_model_window_50np_Opt.py
# This code takes 50 samples and then performs an inference on this data_window.
# As it is a new data_window every 50 samples, the inference time is much slower
# but accuracy is improved.

# predict_nn_model_window_50np_Opt.py WORKS but inference time is 1.5 seconds

import time
import numpy as np
from sense_hat import SenseHat
import tflite_runtime.interpreter as tflite
window_size = 50          #CHANGE IF NEEDED

# Load the TensorFlow Lite model
interpreter = tflite.Interpreter(model_path="nn_model_window_50np_Opt.tflite") #change the name for your model
interpreter.allocate_tensors()

# Get input and output tensors
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Initialize SenseHat
sense = SenseHat()

# Function to get gyro and accel data from SenseHat
def get_sensor_data():
    gyro = sense.get_gyroscope_raw()
    accel = sense.get_accelerometer_raw()
    return [gyro['x'], gyro['y'], gyro['z'], accel['x'], accel['y'], accel['z']]

# Function to prepare data for inference
def prepare_data(window_size): #change window size if needed
    data = []
    for _ in range(window_size):
        sensor_data = get_sensor_data()
        data.append(sensor_data)
        #time.sleep(0.01)  # Adjust the sleep time as needed to match the sampling rate
    return np.array(data).reshape(1, window_size, 6)

# Function to perform inference
def perform_inference():
    input_data = prepare_data()
    interpreter.set_tensor(input_details[0]['index'], input_data.astype(np.float32))
    interpreter.invoke()
    output_data = interpreter.get_tensor(output_details[0]['index'])
    return output_data
# Function to update SenseHat color based on prediction

def update_sensehat_color(prediction):
    # Define colors
    colors = {
        0: (0, 0, 255),    # Blue for sitting
        1: (255, 255, 0),  # Yellow for standing
        2: (0, 255, 0),    # Green for walking
        3: (255, 0, 0)     # Red for falling
    }
    # Get the predicted class
    predicted_class = np.argmax(prediction)
    # Set the SenseHat color
    sense.clear(colors[predicted_class])

# Perform inference and print the result
while True:
    start_time = time.time()
    result = perform_inference()
    end_time = time.time()
    print("Inference result:", result)
    print("Inference time: ",end_time-start_time)
    update_sensehat_color(result)

In [ ]:
# predict_nn_model_window_50np_Opt_SLIDE.py
# This performs a sliding window (using append and pop to the data_window).
# The data_window is essentially an updated buffer and the inference occurs
# at every cycle on whatever data exists. Consistent behavior yields higher
# accuracies.
# predict_tflite_nnn50np3.py WORKS but reduced inference time (0.05s
# but still takes time to do it correctly)
import time
import numpy as np
from sense_hat import SenseHat
import tflite_runtime.interpreter as tflite

# Initialize the sliding window with initial data
window_size = 50              #CHANGE IF NEEDED
data_window = [get_sensor_data() for _ in range(window_size)]

# Load the TensorFlow Lite model
interpreter = tflite.Interpreter(model_path="nn_model_window_50np_Opt.tflite")
interpreter.allocate_tensors()

# Get input and output tensors
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Initialize SenseHat
sense = SenseHat()

# Function to get gyro and accel data from SenseHat
def get_sensor_data():
    gyro = sense.get_gyroscope_raw()
    accel = sense.get_accelerometer_raw()
    return [gyro['x'], gyro['y'], gyro['z'], accel['x'], accel['y'], accel['z']]

# Function to update the sliding window with new data
def update_data_window():
    new_data = get_sensor_data()
    data_window.pop(0)  # Remove the oldest data point
    data_window.append(new_data)  # Append the newest data point

# Function to perform inference
def perform_inference():
    input_data = np.array(data_window).reshape(1, window_size, 6)
    interpreter.set_tensor(input_details[0]['index'], input_data.astype(np.float32))
    interpreter.invoke()
    output_data = interpreter.get_tensor(output_details[0]['index'])
    return output_data

# Function to update SenseHat color based on prediction
def update_sensehat_color(prediction):
    # Define colors
    colors = {
        0: (0, 0, 255),    # Blue for sitting
        1: (255, 255, 0),  # Yellow for standing
        2: (0, 255, 0),    # Green for walking
        3: (255, 0, 0)     # Red for falling
    }
    # Get the predicted class
    predicted_class = np.argmax(prediction)
    # Set the SenseHat color
    sense.clear(colors[predicted_class])

# Perform inference and print the result
while True:
    start_time = time.time()
    update_data_window()  # Update the sliding window with new data
    result = perform_inference()
    end_time = time.time()
    print("Inference result:", result)
    print("Inference time: ", end_time - start_time)
    update_sensehat_color(result)


## EXPLANATION OF DATA INPUT PREPARATION FOR TENSORFLOW LITE MODELS
`input_data = np.array(data_window).reshape(1, window_size, 6):`

*   `np.array(data_window)`: Converts the data_window list into a numpy array. This is necessary because TensorFlow Lite models require arrays/tensors.
*  `.reshape(1, window_size, 6)`: Reshapes the numpy array to have the shape (1, window_size, 6) **where**
*   `1` indicates the batch size
*   `window_size` is the number of time points in the sliding window (50)
*   `6 ` represents the number of features

`interpreter.set_tensor(input_details[0]['index'], input_data.astype(np.float32))`:
*   `interpreter.set_tensor(input_details[0]['index'], ...)`: Sets the input tensor for the TensorFlow Lite interpreter
*  `input_details[0]['index']` specifies the index of the input tensor
*  `input_data.astype(np.float32)`: Converts the input_data to the float32 data type, which is the required format for the model input

`interpreter.invoke()`:
*   This line runs the inference process on the input data that has been set in the TensorFlow Lite interpreter.
*    The interpreter processes the input data through the model and performs the necessary computations to generate the output.

`output_data = interpreter.get_tensor(output_details[0]['index'])`:

*   `interpreter.get_tensor(output_details[0]['index'])`: This retrieves the output tensor from the TensorFlow Lite interpreter
*   `output_details[0]['index']` specifies the index of the output tensor.
*   output_data: This variable stores the `output data` produced by the model after the inference.







# Future Optimization Resources

## Pruning with Keras
https://www.tensorflow.org/model_optimization/guide/pruning/pruning_with_keras

## TF Lite
https://www.tensorflow.org/api_docs/python/tf/lite/Optimize

## Step by Step On-Device Optimization
https://apxml.com/courses/advanced-tensorflow/chapter-6-model-deployment-optimization/optimizing-on-device-inference

